# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

**DOI**: [10.71728/senscience.vq4a-28xa](https://sen.science/doi/10.71728/senscience.vq4a-28xa/)


In [ ]:
# Ensure `mlcroissant` is installed (run once)
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")
print(f"DOI/Identifier: {metadata.identifier}")

## 2. Data Overview
Review available record sets, fields (columns), and their `@id` values.

In [ ]:
from pprint import pprint

# List all record sets
record_sets = [rs for rs in dataset.record_sets]
print(f"Available record sets (by @id): {[rs.id for rs in record_sets]}")

recordset_dict = {rs.id: rs for rs in record_sets}

# Show fields for each record set
for rs in record_sets:
    print(f"\nRecordSet '@id': {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - @id: {field.id}, name: {field.name}, dataType: {field.data_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from above.

> **Tip:** You may inspect the shape and column names of each record set.

In [ ]:
# Collect DataFrames for each record set (by @id)
dataframes = {}
for rs in record_sets:
    print(f"Loading records for RecordSet @id: {rs.id}")
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"Shape: {df.shape}, Columns: {df.columns.tolist()}\n")

# For demonstration, select the main record set (first one) if available
if len(record_sets) > 0:
    selected_rs_id = record_sets[0].id
    display_cols = dataframes[selected_rs_id].columns.tolist()
    print(f"Selected RecordSet '@id': {selected_rs_id}")
    print(f"Columns: {display_cols}")
    dataframes[selected_rs_id].head()
else:
    selected_rs_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps—such as filtering, normalization, grouping—on a numeric field.

In [ ]:
import numpy as np

# Select numeric field by @id (adjust as per available field @id's from previous section)
if selected_rs_id:
    numeric_field_ids = [
        f.id for f in recordset_dict[selected_rs_id].fields 
        if f.data_type in ('schema:Integer', 'schema:Float', 'schema:Number')
    ]
    if len(numeric_field_ids):
        numeric_field_id = numeric_field_ids[0]

        df = dataframes[selected_rs_id].copy()
        print(f"Numeric field selected (@id): {numeric_field_id}\n")

        # Ensure numeric type
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

        threshold = df[numeric_field_id].quantile(0.75)  # Use 75th percentile for demonstration
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (75th percentile): {filtered_df.shape[0]}")
        print(filtered_df[[numeric_field_id]].head())

        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nSample normalized values:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Choose a group field (categorical string field with >1 value)
        string_field_ids = [
            f.id for f in recordset_dict[selected_rs_id].fields
            if f.data_type == 'schema:Text' and df[f.id].nunique() > 1
        ]
        if string_field_ids:
            group_field_id = string_field_ids[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
            print(grouped_df.head())
        else:
            print("No suitable categorical field with >1 category to group by.")
    else:
        print(f"No numeric fields detected in record set {selected_rs_id}.")
else:
    print("No record sets loaded above.")

## 5. Visualization
Visualize the numeric data (e.g., distribution, boxplot, scatterplot if possible).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_rs_id and 'numeric_field_id' in locals():
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[selected_rs_id][numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If group_field_id exists, boxplot
    if 'group_field_id' in locals():
        plt.figure(figsize=(10,5))
        sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} grouped by {group_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print('No numeric field selected or data not available.')

## 6. Conclusion
In this notebook, we demonstrated how to:
- Load and explore a Croissant-standard FAIR dataset with `mlcroissant`
- Inspect record set and field metadata by `@id`
- Extract records from the dataset into pandas DataFrames
- Perform basic EDA and visualize numeric data

You can continue analysis by leveraging `mlcroissant`'s rich metadata, joining across record sets, or integrating external biological resources for in-depth mass spectrometry or protein analysis. For further guidance, consult the documentation at: https://github.com/mlcommons/croissant
